# Rule: **build_biomass_potentials**


**Description**

Compute biogas and solid biomass potentials for each clustered model region using data from JRC ENSPRESO.

The additional configuration parameters associated with this rule are defined under the **biomass** rule:  
- biomass.year  
- biomass.scenario  
- biomass.classes  
- biomass.share_unsustainable_use_retained  
- biomass.share_sustainable_potential_available


**Outputs**

- resources/{prefix}/{name}/`biomass_potentials_all_{clusters}_{planning_horizons}.csv`
- resources/{prefix}/{name}/`biomass_potentials_s_{clusters}_{planning_horizons}.csv`

In [ ]:
######################################## Parameters

### Run
prefix = ''
name = ''

### Network
clusters = '' # number of clusters or 'adm'
opts = ''
sector_opts = ''
horizon = ''

### Spatial domain 'ES' or 'EU' (for maps domain and NUTS regions)
spatial_domain = 'ES'
country_name = 'Spain'

In [ ]:
##### Imports
import pandas as pd
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
pd.set_option("display.max_columns", None)
import os 
import sys
from matplotlib.colors import Normalize

##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp

##### Read params.yaml
params = xp.read_params('../params.yaml')
##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

## `biomass_potentials_all_{clusters}_{planning_horizons}.csv`  
Load the file and preview its content.

In [ ]:
file = f"biomass_potentials_all_{clusters}_{horizon}.csv"

biomass_potentials_all = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

biomass_potentials_all.head()

## `biomass_potentials_s_{clusters}_{planning_horizons}.csv`  
Load the file and preview its content.

In [ ]:
file = f"biomass_potentials_s_{clusters}_{horizon}.csv"

biomass_potentials = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

biomass_potentials.head()
print('Biomass energy potentials in MWh/a')
biomass_potentials.head(5)

### What is the spatial distribution of bioenergy potential across the country?

In [ ]:
# Path to build regions onshore
regions = gpd.read_file(Path(params["rootpath"]) / "resources" / prefix / name / f"regions_onshore_base_s_{clusters}.geojson")

In [ ]:
# Biomass components to plot
biomass_variables = [
    "solid biomass",
    "biogas",
]

titles = {
    "solid biomass": "(a) Solid biomass",
    "biogas": "(b) Biogas",
}

# Convert from MWh/a to TWh/a
biomass_twh = biomass_potentials.copy()
for var in biomass_variables:
    biomass_twh[var] = biomass_twh[var] / 1e6

# Merge with regions
gdf = regions.merge(
    biomass_twh,
    left_on="name",
    right_on=biomass_twh.columns[0],
    how="left"
)

# Common color scale
vmin = gdf[biomass_variables].min().min()
vmax = gdf[biomass_variables].max().max()
norm = Normalize(vmin=vmin, vmax=vmax)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)

for ax, var in zip(axes, biomass_variables):
    gdf.plot(
        column=var,
        cmap="YlGn",
        norm=norm,
        linewidth=0.8,
        ax=ax,
        edgecolor="black",
        legend=False
    )
    ax.set_title(titles[var], fontsize=11)
    ax.axis("off")

# Shared colorbar
sm = plt.cm.ScalarMappable(cmap="YlGn", norm=norm)
sm._A = []

cbar = fig.colorbar(
    sm,
    ax=axes,
    location="right",
    shrink=0.8
)
cbar.set_label("Biomass energy potential (TWh/year)")

fig.suptitle(
    "Biomass energy potentials by model region (Spain)",
    fontsize=13
)

plt.show()